In [123]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

# Notebook is in parkkiko/notebooks/
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_dir = project_root / "data" / "processed"

source_path = processed_dir / "parking_rules.parquet"
rules_output_path = processed_dir / "parking_rules_english.parquet"
classes_output_path = processed_dir / "classes.parquet"


In [124]:
rules = gpd.read_parquet(source_path)

rename_map = {
    "luokka": "class_code",
    "luokka_nimi": "class_name_fi",
    "tyyppi": "parking_type_fi",
    "voimassaolo": "validity_fi",
    "kesto": "duration_text_fi",
    "kausi": "season_text_fi",
    "lisatieto": "extra_info_fi",
}

# Avoid silently creating duplicate columns if both names already exist.
conflicts = [
    (old, new)
    for old, new in rename_map.items()
    if old in rules.columns and new in rules.columns
]
if conflicts:
    raise ValueError(f"Both Finnish and English column names exist: {conflicts}")

rules = rules.rename(
    columns={
        old: new
        for old, new in rename_map.items()
        if old in rules.columns
    }
)

required = [
    "id",
    "class_code",
    "class_name_fi",
    "parking_type_fi",
    "validity_fi",
    "geometry",
]
missing = [col for col in required if col not in rules.columns]

if missing:
    raise KeyError(
        f"Missing columns: {missing}\n"
        f"Available columns: {rules.columns.tolist()}"
    )

codes = pd.to_numeric(rules["class_code"], errors="coerce")

if (rules["class_code"].notna() & codes.isna()).any():
    raise ValueError("class_code contains non-numeric values")

if (codes.dropna() != codes.dropna().round()).any():
    raise ValueError("class_code contains non-integer values")

rules["class_code"] = codes.astype("Int64")

print("Rows:", len(rules))
print("Columns:", rules.columns.tolist())
print("CRS:", rules.crs.to_epsg() if rules.crs is not None else None)

Rows: 8754
Columns: ['id', 'class_code', 'class_name_fi', 'parking_type_fi', 'validity_fi', 'duration_text_fi', 'season_text_fi', 'extra_info_fi', 'geometry', 'rule_type', 'hours', 'duration_min', 'season', 'extra_info', 'status', 'issue_codes', 'reason', 'roadworks_until']
CRS: 3879


In [125]:
A = rules[
    rules["class_code"].isna()
]
A

,id,class_code,class_name_fi,parking_type_fi,validity_fi,duration_text_fi,season_text_fi,extra_info_fi,geometry,rule_type,hours,duration_min,season,extra_info,status,issue_codes,reason,roadworks_until
228,8232,<NA>,NaN,Taxi,,NaN,NaN,"7412, C41, E8","MULTIPOLYGON (((25497531.118 6671727.85, 25497...",reserved,NaN,NaN,NaN,"7412, C41, E8",uncertain,note,the sign carries a condition we do not read; c...,NaT
299,8304,<NA>,NaN,Pysäköintikielto,,,NaN,7639-2,"MULTIPOLYGON (((25492626.605 6671346.41, 25492...",always_banned,NaN,NaN,NaN,7639-2,official,NaN,NaN,NaT
326,8332,<NA>,NaN,Pysäköintikielto,,,NaN,7639-2,"MULTIPOLYGON (((25492447.219 6671457.728, 2549...",always_banned,NaN,NaN,NaN,7639-2,official,NaN,NaN,NaT
336,8342,<NA>,NaN,Pysäköintikielto,,,NaN,7639-1,"MULTIPOLYGON (((25492224.485 6671974.372, 2549...",always_banned,NaN,NaN,NaN,7639-1,official,NaN,NaN,NaT
342,8348,<NA>,NaN,Pysäköintikielto,,,NaN,7639-2,"MULTIPOLYGON (((25492413.644 6671376.002, 2549...",always_banned,NaN,NaN,NaN,7639-2,official,NaN,NaN,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8744,17044,<NA>,NaN,Kuormauspaikka,NaN,NaN,NaN,7836,"MULTIPOLYGON (((25497219.793 6674234.639, 2549...",reserved,NaN,NaN,NaN,7836,official,NaN,NaN,NaT
8747,17047,<NA>,NaN,Pysäyttämiskielto,,NaN,NaN,8062,"MULTIPOLYGON (((25497341.103 6674075.667, 2549...",always_banned,NaN,NaN,NaN,8062,official,NaN,NaN,NaT
8748,17048,<NA>,NaN,Pysäköintikielto,,NaN,NaN,8118,"MULTIPOLYGON (((25496222.316 6674658.409, 2549...",always_banned,NaN,NaN,NaN,8118,official,NaN,NaN,NaT
8749,17049,<NA>,NaN,Taksi,NaN,NaN,NaN,8064,"MULTIPOLYGON (((25496353.602 6673632.915, 2549...",reserved,NaN,NaN,NaN,8064,official,NaN,NaN,NaT


In [126]:
# assign code 0 for 343 rows have both a missing class_code and no Finnish class name
rules["class_code"] = (
    rules["class_code"]
    .fillna(0)
    .astype("Int64")
)

In [127]:
class_translations = {
    1: "Free short-term parking",
    2: "Free long-term parking",
    3: "Single payment, up to 1 hour",
    4: "Single payment, up to 2 hours",
    5: "Single payment, up to 4 hours",
    6: (
        "Paid parking without a resident/business permit; "
        "charging hours vary by location"
    ),
    7: (
        "Single payment, up to 1 hour without a resident/business "
        "permit; charging hours vary by location"
    ),
    8: (
        "Free short-term parking without a resident/business permit; "
        "use a parking disc"
    ),
    9: "Parking allowed outside no-parking hours",
    10: "Paid parking at zone rate",
    11: "Z-permit car-sharing vehicle pickup/return",
}

rules["class_name_en"] = rules["class_code"].map(class_translations)

print(
    rules[["class_code", "class_name_fi", "class_name_en"]]
    .drop_duplicates()
    .sort_values("class_code")
    .to_string(index=False)
)

 class_code                                                                                          class_name_fi                                                                                    class_name_en
          0                                                                                                    NaN                                                                                              NaN
          1                                                                      Ilmainen lyhytaikainen pysäköinti                                                                          Free short-term parking
          2                                                                      Ilmainen pitkäaikainen pysäköinti                                                                           Free long-term parking
          3                                                                            Kertamaksu enintään 1 tunti                                      

In [128]:
parking_type_translations = {
    "Pysäköintikielto": "No parking",
    "pysäköintikielto": "No parking",
    "Pysäyttämiskielto": "No stopping",
    "Sähköpotkulauta": "Electric scooter",
    "Sähköauto": "Electric car",
    "Taxi": "Taxi",
    "Taksi": "Taxi",
    "Taxi, lataus": "Taxi, charging",
    "Kuormauspaikka": "Loading zone",
    "Inva": "Accessible parking",
    "Matkailuliikenne": "Tourist transport",
    "CD": "CD-plate diplomatic vehicles",
    "Moottoripyörä": "Motorcycle",
    "Polkupyörä": "Bicycle",
    "henkilöauto, pakettiauto": "Passenger car or van",
    "Virka-auto": "Official vehicle",
    "Poliisi": "Police",
    "Kaupunginkanslia": "City Executive Office",
    "Kirjastoauto": "Mobile library vehicle",
    "Parklet": "Parklet",
    "Valtioneuvosto": "Finnish Government",
    "Kuorma-auto": "Truck",
}

parking_type_values = rules["parking_type_fi"].astype("string").str.strip()

rules["parking_type_en"] = parking_type_values.map(
    parking_type_translations
)

untranslated = parking_type_values[
    parking_type_values.notna()
    & parking_type_values.ne("")
    & rules["parking_type_en"].isna()
]

print("Non-empty parking types without a verified translation:")
print(untranslated.value_counts().to_string())

Non-empty parking types without a verified translation:
parking_type_fi
9    10
0     5
8     5


In [129]:
classes = (
    rules.loc[
        rules["class_code"].notna(),
        ["class_code", "class_name_fi", "class_name_en"],
    ]
    .groupby("class_code", as_index=False)
    .first()
    .rename(
        columns={
            "class_code": "code",
            "class_name_fi": "name_fi",
            "class_name_en": "name_en",
        }
    )
    .sort_values("code")
    .reset_index(drop=True)
)

classes["code"] = classes["code"].astype("Int64")

print("Classes:", len(classes))
print(classes.to_string(index=False))

Classes: 12
 code                                                                                                name_fi                                                                                          name_en
    0                                                                                                    NaN                                                                                              NaN
    1                                                                      Ilmainen lyhytaikainen pysäköinti                                                                          Free short-term parking
    2                                                                      Ilmainen pitkäaikainen pysäköinti                                                                           Free long-term parking
    3                                                                            Kertamaksu enintään 1 tunti                                                        

In [130]:
if rules["id"].isna().any():
    raise ValueError("Some parking rules have a missing id")

if rules["id"].duplicated().any():
    raise ValueError("Some parking rules have a duplicate id")

if rules.geometry.isna().any():
    raise ValueError("Some parking rules have missing geometry")

if rules.crs is None or rules.crs.to_epsg() != 3879:
    raise ValueError("Expected geometry CRS EPSG:3879")

if not rules.geometry.geom_type.eq("MultiPolygon").all():
    raise ValueError("Expected all geometries to be MultiPolygon")

unmatched_codes = (
    set(rules["class_code"].dropna().tolist())
    - set(classes["code"].tolist())
)

if unmatched_codes:
    raise ValueError(f"Unmatched class codes: {sorted(unmatched_codes)}")

processed_dir.mkdir(parents=True, exist_ok=True)

rules.to_parquet(rules_output_path, index=False)
classes.to_parquet(classes_output_path, index=False)

print("Parking rules:", len(saved_rules))
print("CRS:", saved_rules.crs.to_epsg())

print("Classes:", len(saved_classes))
print("Class columns:", saved_classes.columns.tolist())

print("\nEnglish and cleaned fields in rules:")
print(
    [
        col
        for col in ["class_name_en", "parking_type_en", "validity_clean"]
        if col in saved_rules.columns
    ]
)

Parking rules: 8754
CRS: 3879
Classes: 12
Class columns: ['code', 'name_fi', 'name_en']

English and cleaned fields in rules:
['class_name_en', 'parking_type_en']


In [131]:
rules["reason"].unique()

<ArrowStringArray>
[                                                                                                                           nan,
                                                               'the sign carries a condition we do not read; check it yourself',
                                                                                                        'no rule for class nan',
 'the hours '7-18, Maksullinen (9-18)' also say 'Maksullinen', which we cannot read; unreadable duration 'max 60 min lauantai'',
 'the hours '7-15, Maksullinen (9-18)' also say 'Maksullinen', which we cannot read; unreadable duration 'max 60 min lauantai'',
                                                                      'the class allows 240 min but the register says 1440 min',
                                                                         'the class allows 60 min but the register says 30 min',
                                                                       'the cl

In [132]:
rules["issue_codes"].unique()

<ArrowStringArray>
[nan, 'note', 'unreadable', 'note,unreadable', 'ambiguous']
Length: 5, dtype: str

# Parking rules data model

## Files

- `data/processed/classes.parquet`
- `data/processed/parking_rules_english.parquet`

```mermaid
erDiagram
    CLASSES o|--o{ PARKING_RULES : classifies

    CLASSES {
        int code PK "Parking class code"
        string name_fi "Original Finnish class name"
        string name_en "Reviewed English class name"
    }



    PARKING_RULES {
        int id PK "Parking area ID"
        int class_code FK "Parking class code"
        string issue_codes "Issue code"

        string class_name_fi "Original Finnish class name"
        string class_name_en "Reviewed English class name"

        string parking_type_fi "Original Finnish parking or vehicle type"
        string parking_type_en "English type where translation is verified"

        string validity_fi "Original restriction hours"
        string duration_text_fi "Original Finnish duration text"
        string season_text_fi "Original Finnish seasonal text"
        string extra_info_fi "Original Finnish additional information"

        string rule_type "Cleaned parking-rule category"
        string hours "Cleaned hours value"
        float duration_min "Parsed maximum duration in minutes"
        string season "Structured seasonal rule"
        string extra_info "Cleaned additional information"

        string status "Rule status"
        datetime roadworks_until "Temporary-roadwork end time"

        geometry geometry "MultiPolygon, EPSG:3879"
    }
```


## Data notes
- `PARKING_RULES` contains 8,754 unique records.
- 343 rows have both a missing class_code and no Finnish class name, assign them to code 0
- An uncertain status has different issue_codes.
- An issue code has different reasons.
- Original Finnish source fields are retained for traceability.
- Translated fields support English-language analysis, mapping, and reporting.